In [1]:
import pandas as pd
import numpy as np
import sqlite3
from datasketch import MinHash, MinHashLSH
from collections import Counter
import numpy as np


In [2]:
db_path = "../DatabaseVis/main.db"
conn = sqlite3.connect(db_path)

In [3]:
query = "SELECT rxnorm_ingredient_id, meddra_id, meddra_name FROM meddra_mappings"
df = pd.read_sql_query(query, conn)

In [4]:
drug_adr_groups = df.groupby('rxnorm_ingredient_id')['meddra_id'].apply(list).to_dict()

In [5]:
id_name_dict = dict(zip(df['meddra_id'], df['meddra_name']))

In [6]:

class ADRData:
    def __init__(self, id_to_name_dict):
        """
        Initializes with a dictionary of {meddra_id: meddra_name}.
        """
        self.id_to_name = id_to_name_dict
        self.unique_ids = sorted(list(id_to_name_dict.keys()))
        
        self.id_to_idx = {adr_id: i for i, adr_id in enumerate(self.unique_ids)}
        self.idx_to_id = {i: adr_id for i, adr_id in enumerate(self.unique_ids)}
        
        self.vocab_size = len(self.unique_ids)

    def encode(self, adr_list):
        """
        Takes a list of ADR IDs and returns a binary vector (1s and 0s).
        Example: ['10028553', '10003041'] -> [0, 1, 0, 0, 1...]
        """
        vector = np.zeros(self.vocab_size, dtype=np.int8)
        
        for adr_id in adr_list:
            if adr_id in self.id_to_idx:
                idx = self.id_to_idx[adr_id]
                vector[idx] = 1
            else:
                print(f"Warning: ADR ID {adr_id} not in vocabulary.")
                
        return vector

    def decode(self, vector):
        """
        Takes a binary vector and returns a list of human-readable ADR names.
        """
        decoded_names = []
        
        active_indices = np.where(vector == 1)[0]
        
        for idx in active_indices:
            adr_id = self.idx_to_id[idx]
            name = self.id_to_name.get(adr_id, "Unknown ADR")
            decoded_names.append(name)
            
        return decoded_names


In [7]:
class ADRSignatureDB:
    def __init__(self, num_perm=128, threshold=0.5):
        self.num_perm = num_perm
        self.lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)
        self.signatures = {}
        self.drug_adr_map = {} 
        self.MAX_HASH = 2**64 - 1


    def add_drug_from_vector(self, drug_id, binary_vector, original_adr_ids):

        m = MinHash(num_perm=self.num_perm)
        
        active_indices = np.where(binary_vector == 1)[0]
        
        for idx in active_indices:
            m.update(str(idx).encode('utf8'))
        
        self.lsh.insert(drug_id, m)
        self.signatures[drug_id] = m.hashvalues
        self.drug_adr_map[drug_id] = original_adr_ids

    def get_signature(self, drug_id):
        return self.signatures.get(drug_id).astype(np.uint64)

    def get_normalized_target(self, drug_id):
        raw_sig = self.get_signature(drug_id)

        if raw_sig is None:
            return None

        return (raw_sig.astype(np.float64) / self.MAX_HASH).astype(np.float32)

    def denormalize_prediction(self, predicted_floats):
        clamped = np.clip(predicted_floats, 0, 1)
        return (clamped * self.MAX_HASH).astype(np.uint64)

    def predict_vector(self, predicted_floats, vocab_size, adr_mapping_func=None):

        raw_hashvalues = self.denormalize_prediction(predicted_floats)
        
        m_query = MinHash(num_perm=self.num_perm, hashvalues=raw_hashvalues)
        neighbors = self.lsh.query(m_query)
        print(f"neighbors: {neighbors}, {m_query}")
        reconstructed_vec = np.zeros(vocab_size, dtype=np.int8)
        
        if not neighbors:
            return reconstructed_vec

        for drug_id in neighbors:
            neighbor_adr_ids = self.drug_adr_map.get(drug_id, [])
            for adr_id in neighbor_adr_ids:
                idx = adr_mapping_func(adr_id)
                if idx is not None:
                    reconstructed_vec[idx] = 1
                    
        return reconstructed_vec

In [8]:
adrMapping = ADRData(id_name_dict)
sig_db = ADRSignatureDB(num_perm=128, threshold=0.4)

In [9]:
for rxnorm_id, adr_list in drug_adr_groups.items():
    vec = adrMapping.encode(adr_list)
    
    sig_db.add_drug_from_vector(rxnorm_id, vec, adr_list)

print("ADR Signature Database is ready.")

ADR Signature Database is ready.


In [10]:
data_raw = sig_db.get_signature('1005921')
data_normalized = sig_db.get_normalized_target('1005921')


reconstructed_vec = sig_db.predict_vector(
    data_normalized, 
    vocab_size=adrMapping.vocab_size, 
    adr_mapping_func=adrMapping.id_to_idx.get
)

predicted_names = adrMapping.decode(reconstructed_vec)
print(f"Predicted ADRs: {predicted_names}")

neighbors: ['623400', '1005921'], <datasketch.minhash.MinHash object at 0x00000258C7452B40>
Predicted ADRs: ['Abdominal discomfort', 'Abdominal pain', 'Abnormal sensation in eye', 'Ache', 'Acne', 'Aggression', 'Agitation', 'Agranulocytosis', 'AIDS', 'Alopecia', 'Amenorrhea', 'Angioedema', 'Anxiety', 'Asthenia', 'Ataxia', 'Atrial fibrillation', 'Atrial flutter', 'Atrioventricular block', 'AV block', 'Back pain', 'Blind', 'Blister', 'Bradycardia', 'Breast swelling', 'Breast tenderness', 'Cancer', 'Carcinogenicity', 'Chest pain', 'Chills', 'Confusion', 'Confusional state', 'Constipation', 'Convulsion', 'Coordination abnormal', 'Depression', 'Diarrhea', 'Diarrhoea', 'DIC', 'Diplopia', 'Discomfort', 'Disorientation', 'Disturbance in attention', 'Dizziness', 'Drug hypersensitivity', 'Drunkenness', 'Dry mouth', 'Dry skin', 'Dry throat', 'Dysarthria', 'Dyskinesia', 'Dysmenorrhea', 'Dyspareunia', 'Dyspepsia', 'Ectopic pregnancy', 'Emotional disorder', 'Eosinophilia', 'Epidermal necrolysis', 'Ep

In [11]:
print(data_raw)
print(data_normalized)

[ 48859808  29084147  20108236    901829 123573495      8658  37736748
     21124  44483330  20625186  12721685  42710906   9514205  43543759
  23522020  26659953  15780420  30595717   3764820 119280776 334064348
  64386040  81575942   8152352  24089041 109111922  84314439  14511070
   5852137  21821788   3943068 113470003  24107112   7949399  14140767
  77416038 172691808   8405549 151461289  33856409  48610496  66572828
  39596951   9383822    879113  23512060   8861935 192179014  10985457
   1429394  24582658  57516223   4827103 186977791   9565588  12023628
  14841087 122542078  63238221  41832336  48829588  22750066  15782007
  13760976  58998839 130430973   5447723  48630285  29421555  19214851
  58063191  33091513 117500430  46098326  54321420  62240640  38223933
  36312635  17655973  13536196 255547148 175349571  33674969  58657097
  13487262  22833090  93494019 126057400  17028526  76387787    938489
  48823677  53059232  78906733  29430074  11610906   1881436  66581947
 11819